Step 1: Load and Verify Data

Vibe Check

In [1]:
def glyph_f(x, y):
    return (0 <= x < 2) or (y == 0 and x < 9) or (y == 2 and x < 7)

def glyph_o(x, y):
    in_box = (0 <= x < 8) and (0 <= y < 6)
    is_hole = (1 <= y <= 4) and (2 <= x < 6)
    return in_box and not is_hole

def glyph_r(x, y):
    stem = (0 <= x < 2)
    top_bar = (y == 0 and x < 7)
    right_wall = (y == 1 and 5 <= x < 7)
    mid_bar = (y == 2 and x < 7)
    leg = (y >= 3 and (x == y or x == y + 1))
    return stem or top_bar or right_wall or mid_bar or leg

def glyph_m(x, y):
    left_stem = (0 <= x < 2)
    right_stem = (9 <= x < 11)
    peak = (
        (y == 0 and (x == 2 or 6 <= x < 8)) or
        (y == 1 and (2 <= x < 4 or 6 <= x < 8)) or
        (y == 2 and (3 <= x < 5 or 6 <= x < 8)) or
        (y == 3 and (4 <= x < 7))
    )
    return left_stem or right_stem or peak

def glyph_u(x, y):
    uprights = (0 <= x < 2) or (5 <= x < 7)
    bottom = (y == 5 and 0 <= x < 7)
    return uprights or bottom

def glyph_l(x, y):
    return (0 <= x < 2) or (y == 5 and 0 <= x < 8)

def glyph_a(x, y):
    in_box = (0 <= x < 8) and (0 <= y < 6)
    is_hole = (y in (1, 3, 4, 5)) and (2 <= x < 6)
    return in_box and not is_hole

def glyph_1(x, y):
    serif = (y == 0 and 0 <= x < 5)
    stem = (1 <= y <= 4 and 2 <= x < 5)
    base = (y == 5 and 0 <= x < 8)
    return serif or stem or base


def render_full_f1():
    for r in range(8):
        line = []
        slant_shift = 8 - r
        for c in range(56):
            x = c - slant_shift
            is_slash = (
                (0 <= r <= 2 and 0 <= x < 37) or   
                (0 <= x < 7) or                    
                (4 <= r <= 6 and 9 <= x < 37) or   
                (r == 7 and 9 <= x < 15) or        
                (39 <= x < 46)                    
            )
            line.append("/" if is_slash else " ")
        print("".join(line).rstrip())
    print("\n" * 3)

    letters = [
        (glyph_f, 9),
        (glyph_o, 8),
        (glyph_r, 7),
        (glyph_m, 11),
        (glyph_u, 7),
        (glyph_l, 8),
        (glyph_a, 8),
        (glyph_1, 8),
    ]
    spacings = [1, 1, 1, 1, 1, 1, 6]

    for y in range(6):
        line_chars = [" " * (6 - y)]

        for i, (algo, width) in enumerate(letters):
            for x in range(width):
                line_chars.append("/" if algo(x, y) else " ")

            if i < len(spacings):
                line_chars.append(" " * spacings[i])

        print("".join(line_chars).rstrip())

if __name__ == "__main__":
    render_full_f1()

        /////////////////////////////////////  ///////
       /////////////////////////////////////  ///////
      /////////////////////////////////////  ///////
     ///////                                ///////
    ///////  ////////////////////////////  ///////
   ///////  ////////////////////////////  ///////
  ///////  ////////////////////////////  ///////
 ///////  //////                        ///////




      ///////// //////// /////// ///   // // //   // //       ////////      /////
     //        //    // //   // ////  // // //   // //       //    //        ///
    ///////   //    // /////// // // // // //   // //       ////////        ///
   //        //    // // //   //  ///  // //   // //       //    //        ///
  //        //    // //  //  //       // //   // //       //    //        ///
 //        //////// //   // //       // /////// //////// //    //      ////////


In [2]:
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

DATA = Path('data')
print('working directory:', Path.cwd())
print('files found:', sorted(p.name for p in DATA.glob('*.csv')))

working directory: c:\Users\JamesBundrick\D502
files found: ['circuits.csv', 'constructor_results.csv', 'constructor_standings.csv', 'constructors.csv', 'driver_standings.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']


In [3]:
naive = pd.read_csv(DATA / 'results.csv')
print('dtype of position :', naive['position'].dtype)
print('first values      :', list(naive['position'].head(4)))


dtype of position : str
first values      : ['1', '2', '3', '4']


Missing values writen as the literal two-character string '\N' rather than leaving the field empty. Treat '\N' marker as missing.

In [4]:
results = pd.read_csv(DATA / 'results.csv', na_values=['\\N'])
races   = pd.read_csv(DATA / 'races.csv',   na_values=['\\N'])
status  = pd.read_csv(DATA / 'status.csv',  na_values=['\\N'])

for name, df in [('results', results), ('races', races), ('status', status)]:
    print(f'{name:<8} {df.shape[0]:>7,} rows x {df.shape[1]:>2} columns')

print()
print('dtype of position now :', results['position'].dtype)
print('nulls in position     :', results['position'].isna().sum(), '(the retirements)')

results   26,759 rows x 18 columns
races      1,125 rows x 18 columns
status       139 rows x  2 columns

dtype of position now : float64
nulls in position     : 10953 (the retirements)


Confirm columns. (Fail loudly)

In [5]:
EXPECTED = {
    'results': {'raceId', 'driverId', 'grid', 'position', 'positionOrder', 'statusId'},
    'races':   {'raceId', 'year', 'name'},
    'status':  {'statusId', 'status'},
}
frames = {'results': results, 'races': races, 'status': status}

problems = []
for name, needed in EXPECTED.items():
    missing = needed - set(frames[name].columns)
    print(f"{name:<8} {'all present' if not missing else 'MISSING ' + str(sorted(missing))}")
    if missing:
        problems.append((name, missing))

assert not problems, f'Schema mismatch: {problems}'
print('\nSchema matches the plan.')

results  all present
races    all present
status   all present

Schema matches the plan.


Full column listing.

In [6]:
for name, df in frames.items():
    print(f'--- {name} ---')
    print(list(df.columns))
    print()

--- results ---
['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']

--- races ---
['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time']

--- status ---
['statusId', 'status']



'Finished' and the '+N Lap(s)' entries classified as finishes; everything else is a retirement.

In [7]:
status_counts = (results.merge(status, on='statusId')
                        .groupby('status').size()
                        .sort_values(ascending=False)
                        .rename('rows'))
print(status_counts.to_string())

status
Finished              7674
+1 Lap                4037
Engine                2026
+2 Laps               1613
Accident              1062
Did not qualify       1025
Collision              854
Gearbox                810
Spun off               795
+3 Laps                731
Suspension             431
+4 Laps                405
Did not prequalify     331
Transmission           321
Electrical             316
Brakes                 254
Withdrew               248
+5 Laps                221
Clutch                 214
Not classified         172
Fuel system            155
+6 Laps                153
Disqualified           147
Turbo                  146
Hydraulics             139
Overheating            133
Ignition               128
Oil leak               124
Throttle               111
Retired                107
Out of fuel            100
+7 Laps                100
Halfshaft               99
Wheel                   88
Oil pressure            87
Fuel pump               66
Differential         

Sanity checks.

In [8]:
print('seasons present      :', races['year'].min(), 'to', races['year'].max())
print('races               :', races['raceId'].nunique())
print('driver-race rows    :', len(results))
print('grid == 0 (pit lane):', (results['grid'] == 0).sum())
print('positionOrder nulls :', results['positionOrder'].isna().sum(), '(should be 0)')
print('grid range          :', results['grid'].min(), 'to', results['grid'].max())

seasons present      : 1950 to 2024
races               : 1125
driver-race rows    : 26759
grid == 0 (pit lane): 1638
positionOrder nulls : 0 (should be 0)
grid range          : 0 to 34


In [9]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1
3,4,18,4,4,5.0,11,4.0,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1
4,5,18,5,1,23.0,3,5.0,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1
